## Session 6 — Docker + Cloud Run (Project 2)

**XRO Tech · AI + Google Cloud Architecture Program · Batch 2**

Today you'll package a complete RAG app with Firestore memory into a Docker container and deploy it as Project 2 — your second permanent Cloud Run URL.

Run every cell top to bottom. Read the markdown above each cell before running it.

**What you'll do:**

| Cell | Task |
|---|---|
| 1 | Check your environment |
| 2 | Write the app, requirements, and Dockerfile |
| 3 | Build and push the image |
| 4 | Deploy to Cloud Run |
| 5 | Test cold start vs warm start

**A note on shared infrastructure:** Cell 3 builds a container image using Cloud Build, a separate Google service from this VM. If many students build at the same time, builds can queue — that's expected, not an error.

**Cell 1 — Environment check.**

In [1]:
import os

STUDENT_ID = os.environ.get("STUDENT_ID", "NOT_SET")
BATCH_ID = os.environ.get("BATCH_ID", "NOT_SET")
PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "NOT_SET")
REGION = "asia-south1"
REPO = f"{REGION}-docker.pkg.dev/{PROJECT_ID}/xro-repo"
SERVICE = f"rag-v2-{STUDENT_ID}"

print(f"Student ID : {STUDENT_ID}")
print(f"Batch ID   : {BATCH_ID}")
print(f"Project    : {PROJECT_ID}")
print(f"Image      : {REPO}/project2-{STUDENT_ID}")
print(f"Service    : {SERVICE}")

Student ID : s02
Batch ID   : batch2
Project    : xro-lab
Image      : asia-south1-docker.pkg.dev/xro-lab/xro-repo/project2-s02
Service    : rag-v2-s02


**Cell 2 — Write Project 2: RAG with Firestore memory.** This combines what you built in Session 4 (the RAG pipeline) and Session 5 (Firestore memory) into one Streamlit app, written to a `project2` folder along with its requirements and Dockerfile.

In [2]:
os.makedirs("project2", exist_ok=True)

app_py = '''import streamlit as st
import os
import json
import time
import uuid
import numpy as np
import vertexai
from vertexai.language_models import TextEmbeddingModel
from vertexai.generative_models import GenerativeModel
from google.cloud import storage, firestore

PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "xro-lab")
STUDENT_ID = os.environ.get("STUDENT_ID", "student")
BATCH_ID = os.environ.get("BATCH_ID", "batch2")
REGION = "asia-south1"

st.set_page_config(page_title="XRO Project 2", layout="wide")


@st.cache_resource
def load_all():
    vertexai.init(project=PROJECT_ID, location=REGION)
    em = TextEmbeddingModel.from_pretrained("text-embedding-004")
    gm = GenerativeModel("gemini-2.5-flash")
    db = firestore.Client(project=PROJECT_ID)
    gcs = storage.Client(project=PROJECT_ID)
    blob = gcs.bucket("xro-lab-data").blob(f"{BATCH_ID}/{STUDENT_ID}/session3_vector_index.json")
    idx = json.loads(blob.download_as_text())["chunks"]
    return em, gm, db, idx


em, gm, db, vector_index = load_all()


def cosine_sim(v1, v2):
    a, b = np.array(v1), np.array(v2)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))


def get_history(sid):
    docs = (
        db.collection("conversations")
        .document(f"{STUDENT_ID}_{sid}")
        .collection("messages")
        .order_by("timestamp", direction=firestore.Query.DESCENDING)
        .limit(5)
        .stream()
    )
    return list(reversed([{"role": d.get("role"), "content": d.get("content")} for d in docs]))


def save_msg(sid, role, content):
    db.collection("conversations").document(f"{STUDENT_ID}_{sid}").collection("messages").document().set({
        "role": role,
        "content": content,
        "timestamp": firestore.SERVER_TIMESTAMP,
    })


def rag(question, sid):
    qv = em.get_embeddings([question])[0].values
    chunks = sorted(vector_index, key=lambda c: cosine_sim(qv, c["vector"]), reverse=True)[:3]
    ctx = "\\n\\n".join([f"[{i + 1}] {c[\'text\']}" for i, c in enumerate(chunks)])
    hist = "\\n".join([f"{m[\'role\'].upper()}: {m[\'content\']}" for m in get_history(sid)])
    prompt = f"Context:\\n{ctx}\\n\\nHistory:\\n{hist}\\n\\nQ: {question}\\nA:"
    ans = gm.generate_content(prompt).text
    save_msg(sid, "user", question)
    save_msg(sid, "assistant", ans)
    return ans


st.title("XRO RAG Assistant -- Project 2")
st.caption(f"Student: {STUDENT_ID} | With Firestore memory")

if "session_id" not in st.session_state:
    st.session_state.session_id = str(uuid.uuid4())[:8]
sid = st.session_state.session_id
st.caption(f"Session: {sid}")

history = get_history(sid)
for msg in history:
    st.chat_message(msg["role"]).write(msg["content"])

if prompt := st.chat_input("Ask about the document..."):
    st.chat_message("user").write(prompt)
    with st.spinner("Thinking..."):
        answer = rag(prompt, sid)
    st.chat_message("assistant").write(answer)
'''

with open("project2/app.py", "w") as f:
    f.write(app_py)

requirements_txt = (
    "streamlit==1.32.0\n"
    "google-cloud-aiplatform==1.48.0\n"
    "google-cloud-storage==2.16.0\n"
    "google-cloud-firestore==2.16.0\n"
    "numpy==1.26.4\n"
)
with open("project2/requirements.txt", "w") as f:
    f.write(requirements_txt)

dockerfile = (
    "FROM python:3.11-slim\n"
    "WORKDIR /app\n"
    "COPY requirements.txt .\n"
    "RUN pip install --no-cache-dir -r requirements.txt\n"
    "COPY . .\n"
    "EXPOSE 8080\n"
    'CMD ["streamlit", "run", "app.py", "--server.port=8080", "--server.address=0.0.0.0"]\n'
)
with open("project2/Dockerfile", "w") as f:
    f.write(dockerfile)

print("Project 2 files written:")
for fname in ["app.py", "requirements.txt", "Dockerfile"]:
    size = os.path.getsize(f"project2/{fname}")
    print(f"  project2/{fname}  ({size} bytes)")

Project 2 files written:
  project2/app.py  (2897 bytes)
  project2/requirements.txt  (124 bytes)
  project2/Dockerfile  (216 bytes)


**Cell 3 — Build and push to Artifact Registry.** Open a terminal (File → New → Terminal) and run these one at a time. The repository was already created in Session 4, so you won't need to create it again.

In [3]:
print(f"""
Run these in a terminal, one at a time:

cd project2

gcloud builds submit \\
  --tag {REPO}/project2-{STUDENT_ID} .

# Build takes 3-5 minutes. Watch progress at:
# GCP Console -> Cloud Build -> History
#
# If gcloud shows a log-streaming error after "Uploading tarball", that is expected --
# check real status anytime with: gcloud builds list --project={PROJECT_ID} --limit=3
""")


Run these in a terminal, one at a time:

cd project2

gcloud builds submit \
  --tag asia-south1-docker.pkg.dev/xro-lab/xro-repo/project2-s02 .

# Build takes 3-5 minutes. Watch progress at:
# GCP Console -> Cloud Build -> History
#
# If gcloud shows a log-streaming error after "Uploading tarball", that is expected --
# check real status anytime with: gcloud builds list --project=xro-lab --limit=3



**Cell 4 — Deploy to Cloud Run.**

In [4]:
print(f"""
gcloud run deploy {SERVICE} \\
  --image {REPO}/project2-{STUDENT_ID} \\
  --region {REGION} \\
  --allow-unauthenticated \\
  --memory 512Mi \\
  --concurrency 80 \\
  --max-instances 5 \\
  --set-env-vars STUDENT_ID={STUDENT_ID},BATCH_ID={BATCH_ID},GOOGLE_CLOUD_PROJECT={PROJECT_ID}

# Allow public access explicitly
# (--allow-unauthenticated above does not always apply the binding automatically on this project)
gcloud run services add-iam-policy-binding {SERVICE} \\
  --region {REGION} --member=allUsers --role=roles/run.invoker

# Get your URL
gcloud run services describe {SERVICE} --region {REGION} --format="value(status.url)"
""")


gcloud run deploy rag-v2-s02 \
  --image asia-south1-docker.pkg.dev/xro-lab/xro-repo/project2-s02 \
  --region asia-south1 \
  --allow-unauthenticated \
  --memory 512Mi \
  --concurrency 80 \
  --max-instances 5 \
  --set-env-vars STUDENT_ID=s02,BATCH_ID=batch2,GOOGLE_CLOUD_PROJECT=xro-lab

# Allow public access explicitly
# (--allow-unauthenticated above does not always apply the binding automatically on this project)
gcloud run services add-iam-policy-binding rag-v2-s02 \
  --region asia-south1 --member=allUsers --role=roles/run.invoker

# Get your URL
gcloud run services describe rag-v2-s02 --region asia-south1 --format="value(status.url)"



**Cell 5 — Cold start vs warm start.** Paste your Cloud Run URL below. The first request after a period of inactivity is slower because Cloud Run has to start a new container instance — this is called a cold start.

In [6]:
import urllib.request
import time

CLOUD_RUN_URL = "https://rag-v2-s02-220325503612.asia-south1.run.app/"

if "YOUR-SERVICE" in CLOUD_RUN_URL:
    print("Replace CLOUD_RUN_URL with your actual URL first.")
else:
    t0 = time.time()
    urllib.request.urlopen(CLOUD_RUN_URL, timeout=30)
    cold_start = round(time.time() - t0, 2)
    print(f"First request : {cold_start}s")

    t1 = time.time()
    urllib.request.urlopen(CLOUD_RUN_URL, timeout=10)
    warm = round(time.time() - t1, 2)
    print(f"Second request: {warm}s")
    print(f"Speedup       : {round(cold_start / warm, 1)}x faster when warm")

First request : 0.1s
Second request: 0.07s
Speedup       : 1.4x faster when warm


---

### Session 6 complete — Project 2 deployed

You containerized a full RAG app with persistent memory and deployed it to Cloud Run, with auto-scaling and a permanent HTTPS URL.

**Before you close:** save the notebook, and commit to GitHub if you're tracking your work there.

**Next session:** Session 7 builds a REST API backend with FastAPI and adds AI function calling.